# Notebook B — SemSeg + transfer learning for environment classification

**Task:** same multi-label environment classification as Notebook A
(`forest, open_field, water, city`), but via **semantic segmentation + transfer
learning**: fine-tune a pretrained SegFormer head on the 4 environment classes, then derive the
per-frame multi-label by thresholding each class's **pixel-area fraction**.

Outputs per-frame predictions to `dataset/eval/env_pred_semseg.csv` and runtime/frame to
`dataset/eval/runtime_semseg.json`, in the same format as Notebook A for `seg_evaluation.ipynb`.


## 0. Dependencies
Fine-tuning needs `datasets` + `accelerate` (run once if missing).

In [1]:
# !pip install accelerate
# (ADE20K now loads from the official CSAIL zip, so the `datasets` package is NOT required.)
print("If the Trainer import fails, uncomment the pip line above and re-run.")

If the Trainer import fails, uncomment the pip line above and re-run.


## 1. Setup

In [ ]:
import sys, json, time
from pathlib import Path

import numpy as np
import cv2
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
import segmentation_common as sc

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
ENV_CLASSES = sc.CATEGORIES["environment"]
ENV_ID = {c: i for i, c in enumerate(ENV_CLASSES)}   # env-only label space 0..3
print("Device:", DEVICE, "| environment classes:", ENV_CLASSES)
if DEVICE == "cpu":
    print("WARNING: training on CPU is slow - keep MAX_STEPS small for a smoke run.")

## 2. Configuration

In [ ]:
BASE_MODEL = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"
DATASET = "MAPILLARY"                     # "MAPILLARY" (POV-matched, street-level) | "ADE20K"
MAPILLARY_ROOT = Path("../dataset/external/mapillary_vistas")
OUTPUT_DIR = Path("../models/segformer_env")

# --- Test set: the hand-labeled eval data. NEVER train on these images. ---
# build_test_dataset.py drew some Mapillary images INTO this test set, so we
# read candidates.csv below and hold those exact images out of training to keep
# the zero-shot vs. fine-tuned comparison leakage-free.
TEST_IMAGES = Path("../dataset/test_images")
CANDIDATES_CSV = TEST_IMAGES / "candidates.csv"
TEST_LABELS_CSV = TEST_IMAGES / "labels.csv"

NUM_EPOCHS = 5
BATCH_SIZE = 4
LR = 6e-5
MAX_STEPS = None                         # set e.g. 50 for a quick smoke run
AREA_THRESHOLD = 0.03                    # class present if it covers >3% of the frame
TRAIN_LIMIT = None                       # cap #training images (e.g. 1500 to go faster); None = all
VAL_LIMIT = 200                          # #images for in-training mIoU monitoring

PRED_CSV = Path("../dataset/eval/env_pred_semseg.csv")
RUNTIME_JSON = Path("../dataset/eval/runtime_semseg.json")


## 3. Label harmonization -> environment-only label space

Source class names are remapped to the 5 environment ids; everything else -> VOID (ignored).

In [ ]:
def ade20k_source_names():
    from huggingface_hub import hf_hub_download
    import json as _json
    p = hf_hub_download("huggingface/label-files", "ade20k-id2label.json", repo_type="dataset")
    id2label = _json.load(open(p))
    n = max(int(k) for k in id2label) + 1
    names = ["other"] * (n + 1)          # ADE masks are 1-indexed; 0 == other
    for k, v in id2label.items():
        names[int(k) + 1] = v
    return names


if DATASET == "ADE20K":
    source_names = ade20k_source_names()
    LUT = sc.build_id_lookup(source_names, sc.ADE20K_TO_TAXONOMY, class_id=ENV_ID)
elif DATASET == "MAPILLARY":
    import json as _json
    cfg = _json.load(open(MAPILLARY_ROOT / "config_v2.0.json"))
    source_names = [lbl["name"] for lbl in cfg["labels"]]
    LUT = sc.build_id_lookup(source_names, sc.MAPILLARY_TO_TAXONOMY, class_id=ENV_ID)
else:
    raise ValueError(DATASET)

print(f"{DATASET}: {int((LUT != sc.VOID_ID).sum())} source classes map into the {len(ENV_CLASSES)} env classes")

## 4. Dataset & preprocessing

In [ ]:
from torch.utils.data import Dataset
from transformers import SegformerImageProcessor
from PIL import Image
import urllib.request, zipfile

processor = SegformerImageProcessor.from_pretrained(BASE_MODEL, do_reduce_labels=False)


class SegDataset(Dataset):
    def __init__(self, items):
        self.items = items                # list of (load_img, load_mask) callables

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        load_img, load_mask = self.items[i]
        mask = LUT[load_mask()].astype(np.uint8)             # remap to env ids
        enc = processor(load_img(), mask, return_tensors="pt")
        return {"pixel_values": enc["pixel_values"][0], "labels": enc["labels"][0]}


# ADE20K via the official CSAIL zip (script-free: recent `datasets` dropped loading scripts,
# so `load_dataset("scene_parse_150", trust_remote_code=True)` no longer works).
ADE_URL = "http://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip"
ADE_ROOT = Path("../dataset/external/ADEChallengeData2016")


def ensure_ade20k() -> Path:
    if ADE_ROOT.exists():
        return ADE_ROOT
    ADE_ROOT.parent.mkdir(parents=True, exist_ok=True)
    zip_path = ADE_ROOT.parent / "ADEChallengeData2016.zip"
    if not zip_path.exists():
        print("Downloading ADE20K (~1 GB, one-time)...")
        urllib.request.urlretrieve(ADE_URL, zip_path)
    print("Extracting...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(ADE_ROOT.parent)
    return ADE_ROOT


def build_ade20k_items(split, limit=None):        # split: "training" | "validation"
    # ADE annotation PNGs are single-channel with pixel value == class id (0=other, 1..150),
    # which lines up with LUT built from ade20k_source_names().
    root = ensure_ade20k()
    img_dir, ann_dir = root / "images" / split, root / "annotations" / split
    imgs = sorted(img_dir.glob("*.jpg"))[:limit]
    return [(lambda p=p: np.array(Image.open(p).convert("RGB")),
             lambda a=ann_dir / (p.stem + ".png"): np.array(Image.open(a))) for p in imgs]


# --- Mapillary Vistas: POV-matched street-level source (default) ---------------
def mapillary_test_stems(candidates_csv):
    """Basenames of Mapillary images that are already in the hand-labeled test
    set. Held out of training so the test-set comparison stays leakage-free.
    (Image-level hold-out; Vistas frames are sampled sparsely, so near-duplicate
    leakage is minor - unlike dense ride video.)"""
    import csv as _csv
    stems = set()
    if Path(candidates_csv).exists():
        for row in _csv.DictReader(open(candidates_csv)):
            if row.get("source") == "mapillary":
                stems.add(Path(row["orig_path"]).stem)
    return stems


def _mapillary_label_dir(split):
    for d in (MAPILLARY_ROOT / split / "v2.0" / "labels",
              MAPILLARY_ROOT / split / "labels"):
        if d.exists():
            return d
    raise FileNotFoundError(f"no Mapillary labels found for split {split!r}")


def load_vistas_mask(p):
    # same read as build_test_dataset.py: pixel value == class id (config order)
    arr = np.array(Image.open(p))
    return arr[..., 0] if arr.ndim == 3 else arr


def build_mapillary_items(split, exclude_stems=frozenset(), limit=None):
    img_dir = MAPILLARY_ROOT / split / "images"
    lbl_dir = _mapillary_label_dir(split)
    items = []
    for img in sorted(img_dir.glob("*.jpg")):
        if img.stem in exclude_stems:
            continue
        mask = lbl_dir / f"{img.stem}.png"
        if not mask.exists():
            continue
        items.append((lambda p=img: np.array(Image.open(p).convert("RGB")),
                      lambda a=mask: load_vistas_mask(a)))
        if limit and len(items) >= limit:
            break
    return items


if DATASET == "ADE20K":
    # Tip: pass limit=200 for a quick smoke run before the full 20k-image train set.
    train_ds = SegDataset(build_ade20k_items("training"))
    val_ds = SegDataset(build_ade20k_items("validation"))
    print("train/val sizes:", len(train_ds), len(val_ds))
elif DATASET == "MAPILLARY":
    held_out = mapillary_test_stems(CANDIDATES_CSV)
    # train on the 18k training split MINUS every test-set image; monitor mIoU
    # on a slice of the validation split, also test-excluded.
    train_ds = SegDataset(build_mapillary_items("training", held_out, limit=TRAIN_LIMIT))
    val_ds = SegDataset(build_mapillary_items("validation", held_out, limit=VAL_LIMIT))
    print(f"held out {len(held_out)} test-set Mapillary images from training")
    print("train/val sizes:", len(train_ds), len(val_ds))

## 5. Model (transfer learning: new 5-class head)

In [ ]:
from transformers import SegformerForSemanticSegmentation

id2label = {i: c for c, i in ENV_ID.items()}
model = SegformerForSemanticSegmentation.from_pretrained(
    BASE_MODEL,
    num_labels=len(ENV_CLASSES),
    id2label=id2label,
    label2id=ENV_ID,
    ignore_mismatched_sizes=True,        # replace the Cityscapes head with a fresh env-class one
).to(DEVICE)
print("env loss ignore_index:", model.config.semantic_loss_ignore_index, "(== VOID 255)")

## 6. Train

In [7]:
from transformers import TrainingArguments, Trainer


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = torch.tensor(logits)
    up = F.interpolate(logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
    preds = up.argmax(1).numpy()
    cm = np.zeros((len(ENV_CLASSES), len(ENV_CLASSES)), np.int64)
    for p, g in zip(preds, labels):
        valid = g != sc.VOID_ID
        idx = g[valid] * len(ENV_CLASSES) + p[valid]
        cm += np.bincount(idx, minlength=len(ENV_CLASSES) ** 2).reshape(cm.shape)
    tp = np.diag(cm); denom = cm.sum(0) + cm.sum(1) - tp
    iou = np.where(denom > 0, tp / np.clip(denom, 1, None), np.nan)
    return {"mIoU": float(np.nanmean(iou))}


args = TrainingArguments(
    output_dir=str(OUTPUT_DIR), learning_rate=LR, num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS if MAX_STEPS else -1,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="mIoU", greater_is_better=True, logging_steps=20,
    remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, compute_metrics=compute_metrics)
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
processor.save_pretrained(str(OUTPUT_DIR))
print("Saved fine-tuned env model to", OUTPUT_DIR)

/Users/hendrickfischer/Documents/Education/CAS_Master/Vorlesungen/Semester_2/Bildverarbeitung/RRCP_Project/computer_vision_bikeability_score/CRSvenv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

## 7. SemSeg -> multi-label classifier (area threshold)

In [ ]:
@torch.no_grad()
def segment_env(image_rgb: np.ndarray) -> np.ndarray:
    enc = processor(image_rgb, return_tensors="pt").to(DEVICE)
    logits = model(**enc).logits
    up = F.interpolate(logits, size=image_rgb.shape[:2], mode="bilinear", align_corners=False)
    return up.argmax(1)[0].to(torch.uint8).cpu().numpy()


def classify_environment_semseg(image_rgb: np.ndarray, area_threshold: float = AREA_THRESHOLD) -> dict:
    """Multi-label prediction: a class is present if it covers > area_threshold of the frame."""
    mask = segment_env(image_rgb)
    n = mask.size
    return {c: int((mask == ENV_ID[c]).sum() / n > area_threshold) for c in ENV_CLASSES}

## 8. Predict over the test set + runtime

In [ ]:
def run_semseg_testset() -> pd.DataFrame:
    # Recurse the labeled test set (ade20k/, mapillary/, own_frames/); the
    # 'filename' column matches labels.csv so seg_evaluation.ipynb can join them.
    exts = {".jpg", ".jpeg", ".png"}
    imgs = sorted(p for p in TEST_IMAGES.rglob("*")
                  if p.suffix.lower() in exts and "overview" not in p.parts)
    if not imgs:
        print("No test images found - run scripts/build_test_dataset.py first.")
        return pd.DataFrame()

    rows, t0 = [], time.perf_counter()
    for fp in imgs:
        img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
        rel = str(fp.relative_to(TEST_IMAGES))
        rows.append({"filename": rel, **classify_environment_semseg(img)})
    ms = (time.perf_counter() - t0) / len(imgs) * 1000

    df = pd.DataFrame(rows)
    PRED_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(PRED_CSV, index=False)
    json.dump({"method": "semseg", "ms_per_frame": ms, "n": len(imgs)}, open(RUNTIME_JSON, "w"))
    print(f"Saved {len(df)} predictions -> {PRED_CSV}  |  {ms:.1f} ms/frame")
    return df


predictions = run_semseg_testset()
predictions.head()